# 📦 Instalacja wymaganych bibliotek

**Instalujemy:**
- **`lyricsgenius`** – klient API Genius, który pozwoli nam pobierać teksty utworów.
- **`scikit-learn`** – do podziału danych na zbiór treningowy i testowy (`train_test_split`).
- **`pandas`** – do wygodnej manipulacji i analizy danych w DataFrame'ach.
- **`tqdm`** – do wyświetlania wizualnych pasków postępu podczas długotrwałych pętli.
- **`ipywidgets`** – do renderowania widżetów w Jupyter.

In [1]:
# %pip install -q lyricsgenius scikit-learn pandas tqdm ipywidgets

# ⚙️ Konfiguracja i importy – lista artystów i zmienne środowiskowe

W tej komórce przygotowujemy całą konfigurację projektu:

**Token API Genius** – pobierany ze zmiennej środowiskowej `GENIUS_ACCESS_TOKEN`. Aby użyć tego kodu, musisz wygenerować własny token na stronie [Genius API Clients](https://genius.com/api-clients). Jeśli zmienna nie jest ustawiona, `os.environ.get()` zwróci `None`, co później spowoduje błąd autoryzacji.

**Lista raperów (`ARTISTS`)** – 18 polskich artystów, których teksty będziemy analizować. Są to przedstawiciele głównie gatunków rap i hiphop, ale różnych stylów i pokoleń – od bardziej mainstreamowych (Mata, Quebonafide, Pezet) po undergroundowych (Diho, Sentino). Taki przekrój pomoże modelowi nauczyć się rozróżniać charakterystyczne style językowe.

**Parametry:**
- `MAX_SONGS_PER_ARTIST = 50` – maksymalna liczba utworów do pobrania na artystę (50 najpopularniejszych).
- `PAUSE_BETWEEN_ARTISTS = 5` – dodatkowe sekundy przerwy między artystami, by nie narazić się na blokadę API.
- `LYRICS_DIR = "dataset/lyrics"` – katalog, w którym cache'ujemy surowe odpowiedzi API (jeden plik JSON na artystę).

In [2]:
import os

# Token z https://genius.com/api-clients (Client Access Token)
GENIUS_ACCESS_TOKEN = os.environ.get("GENIUS_ACCESS_TOKEN")

# Polscy raperzy 
ARTISTS = [
    "Pezet",
    "Oki",
    "White 2115",
    "Bedoes 2115",
    "Malik Montana",
    "Żabson",
    "Mata",
    "Zeamsone",
    "Young Igi",
    "Sobel",
    "Young Leosia",
    "Quebonafide",
    "Guzior",
    "Avi (POL)",
    "OG Olgierd",
    "Diho",
    "Sentino",
    "Kabe"
]

MAX_SONGS_PER_ARTIST = 50   # Maksymalna liczba piosenek do pobrania dla każdego artysty
PAUSE_BETWEEN_ARTISTS = 5   # Sekundy dodatkowej przerwy między artystami

DATASET_DIR = "dataset"
LYRICS_DIR = "dataset/lyrics"
os.makedirs(LYRICS_DIR, exist_ok=True)

# 📥 Pobieranie utworów z API Genius – funkcja z cache'owaniem

Sercem tego etapu jest funkcja `fetch_artist_songs`, która:

1. **Sprawdza cache** – jeśli plik JSON dla danego artysty już istnieje w `dataset/lyrics/`, ładuje go z dysku i od razu zwraca. Dzięki temu przy ponownym uruchomieniu notebooka nie musimy czekać na API.

2. **Pobiera dane z Genius API** – wywołuje `genius.search_artist()` z biblioteki `lyricsgenius`. Wyniki sortujemy według popularności, co daje nam najważniejsze utwory danego artysty.

3. **Obsługuje błędy** – mamy trzy warstwy zabezpieczeń:
   - **HTTP 429 (rate limit)** – czeka 60 sekund i próbuje ponownie (do 3 razy).
   - **Inne błędy sieciowe** (timeout, brak połączenia) – czeka 30 sekund i próbuje ponownie.
   - **Pętla `for/else`** – jeśli wszystkie 3 próby się nie powiodą, wypisuje komunikat i zwraca pustą listę, zamiast przerywać skrypt.

4. **Zapisuje do cache** – po udanym pobraniu zapisuje JSON ze strukturą: `[{"title": ..., "primary_artist": ..., "lyrics": ...}, ...]`.

**Ważne ustawienia `Genius()`:**
- `remove_section_headers=False` – zachowujemy oryginalne nagłówki sekcji (np. `[Zwrotka 1: Mata]`), które będą kluczowe do przypisania wykonawców.
- `excluded_terms` – odrzucamy remiksy, wersje live i demo, które mogłyby wprowadzić szum do danych.

In [3]:
import json
import re
import time
from pathlib import Path

import lyricsgenius
import requests

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=20,
    sleep_time=0.01, # Przerwa między zapytaniami - ochrona przed rate limitem
    retries=3,
    remove_section_headers=False, # Nagłówki (np. [Zwrotka 3: Taco Hemingway]) potrzebne do etykiet
    skip_non_songs=True,
    excluded_terms=["(Remix)", "(Live)", "(Snippet)", "(Demo)", "(Tracklista)"],
)


def cache_path(artist_name):
    safe_name = re.sub(r"[^\w]+", "_", artist_name).strip("_")
    return Path(LYRICS_DIR) / f"{safe_name}.json"


def fetch_artist_songs(artist_name, max_songs):
    """Pobiera utwory artysty z Genius API (z cachem na dysku i obsługą błędów)."""
    cache = cache_path(artist_name)
    if cache.exists():
        songs = json.loads(cache.read_text(encoding="utf-8"))
        print(f"[cache] {artist_name}: {len(songs)} utworów")
        return songs

    for attempt in range(1, 4):
        try:
            artist = genius.search_artist(
                artist_name,
                max_songs=max_songs,
                sort="popularity",
                allow_name_change=False,
            )
            break
        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 429:
                print(f"[429] {artist_name}: rate limit, czekam 60 s (próba {attempt}/3)...")
                time.sleep(60)
            else:
                raise
        except requests.exceptions.RequestException as e:
            print(f"[błąd sieci] {artist_name}: {e}, czekam 30 s (próba {attempt}/3)...")
            time.sleep(30)
    else:
        print(f"[pominięto] {artist_name}: nie udało się pobrać danych")
        return []

    songs = [
        {"title": s.title, "primary_artist": s.artist, "lyrics": s.lyrics}
        for s in (artist.songs if artist else [])
        if s.lyrics
    ]
    cache.write_text(json.dumps(songs, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[API] {artist_name}: {len(songs)} utworów (zapisano do cache)")
    return songs

# ▶️ Pobieranie danych wszystkich artystów – pętla główna

Wykonujemy właściwe pobieranie: przechodzimy przez wszystkich 18 artystów z listy `ARTISTS` i dla każdego wywołujemy `fetch_artist_songs`.

**Pasek postępu `tqdm`** – dzięki temu zobaczymy elegancki widżet z estymowanym czasem końca.

**Mechanizm cache'owania:** jeśli dany artysta był już wcześniej pobrany (plik cache istnieje), funkcja zwróci dane z dysku – nie odbijamy się wtedy od API i nie czekamy dodatkowych 5 sekund. Dzięki temu ponowne uruchomienie tej komórki jest błyskawiczne.

Po zakończeniu pętli wyświetlamy łączną liczbę pobranych utworów – to wstępna informacja, zanim jeszcze przystąpimy do dzielenia na zwrotki.

In [4]:
from tqdm.notebook import tqdm

all_songs = []
for artist_name in tqdm(ARTISTS, desc="Pobieranie utworów"):
    was_cached = cache_path(artist_name).exists()
    songs = fetch_artist_songs(artist_name, MAX_SONGS_PER_ARTIST)
    all_songs.extend(songs)
    if not was_cached:
        time.sleep(PAUSE_BETWEEN_ARTISTS) # Dodatkowa przerwa między artystami

print(f"\nŁącznie pobrano {len(all_songs)} utworów")

Pobieranie utworów:   0%|          | 0/18 [00:00<?, ?it/s]

[cache] Pezet: 50 utworów
[cache] Oki: 50 utworów
[cache] White 2115: 50 utworów
[cache] Bedoes 2115: 50 utworów
[cache] Malik Montana: 50 utworów
[cache] Żabson: 50 utworów
[cache] Mata: 50 utworów
[cache] Zeamsone: 50 utworów
[cache] Young Igi: 50 utworów
[cache] Sobel: 50 utworów
[cache] Young Leosia: 49 utworów
[cache] Quebonafide: 50 utworów
[cache] Guzior: 50 utworów
[cache] Avi (POL): 50 utworów
[cache] OG Olgierd: 18 utworów
[cache] Diho: 50 utworów
[cache] Sentino: 50 utworów
[cache] Kabe: 50 utworów

Łącznie pobrano 867 utworów


# ✂️ Dzielenie tekstów na zwrotki – parsing i ekstrakcja sekcji

Ten blok tnie surowe teksty utworów na pojedyncze zwrotki i do każdej z nich przypisuje wykonawców. To najważniejszy etap przygotowania danych – jakość tego parsowania bezpośrednio wpływa na skuteczność modelu.

**Definiujemy trzy główne narzędzia:**

1. **`HEADER_RE`** – wyrażenie regularne wykrywające nagłówki sekcji w formacie `[Zwrotka 1: Mata]`, `[Refren]`, `[Bridge]` itp. To one wyznaczają granice między kolejnymi częściami utworu.

2. **`KEEP_SECTIONS`** – lista typów sekcji, które chcemy zachować. Uwzględniamy zarówno polskie nazwy (zwrotka, refren, przejście) jak i angielskie (verse, chorus, bridge).

3. **`ARTIST_SPLIT_RE`** – regex do rozdzielania wykonawców w nagłówkach. Obsługuje separatory: `&`, `,`, `+`, `/`, `i`, `oraz`, `x`, `feat.`, `ft.` – dzięki czemu poprawnie rozpoznamy np. `Mata & Quebonafide` jako dwóch osobnych wykonawców.

**Kluczowe funkcje:**

- **`clean_lyrics()`** – usuwa śmieci dodane przez Genius: nagłówek strony, stopkę "Embed" oraz "You might also like".
- **`parse_header()`** – z nagłówka wyciąga typ sekcji (np. "zwrotka" z "Zwrotka 2") oraz listę wykonawców.
- **`split_into_verses()`** – główna funkcja łącząca wszystko: dla każdej sekcji wycina tekst, parsuje nagłówek, filtruje niechciane typy i zbyt krótkie fragmenty (`MIN_WORDS_PER_VERSE = 10`), a na końcu zwraca listę słowników z polami: `song`, `section`, `text` i `label`.

**Przykład:** Zwrotka z nagłówkiem `[Zwrotka 1: Taco Hemingway & Quebonafide]` zostanie podzielona na jeden rekord z `label = "Quebonafide, Taco Hemingway"`.

In [5]:
# Nagłówek sekcji, np. [Zwrotka 1: Taco Hemingway & Quebonafide]
HEADER_RE = re.compile(r"\[([^\[\]\n]+)\]")

# Typy sekcji, które traktujemy jako "zwrotki" do klasyfikacji
KEEP_SECTIONS = ("zwrotka", "verse", "refren", "chorus", "hook", "bridge", "intro", "outro", "pre-chorus", "prechorus", "przejście", "part", "część")

# Separatory wykonawców w nagłówku: "A & B", "A, B", "A i B", "A feat. B", "A x B"...
ARTIST_SPLIT_RE = re.compile(
    r"\s*(?:,|&|\+|/|\bi\b|\boraz\b|\bx\b|\bfeat\.?\b|\bft\.?\b)\s*",
    re.IGNORECASE,
)

MIN_WORDS_PER_VERSE = 10  # Odrzucamy zbyt krótkie sekcje


def clean_lyrics(raw_lyrics):
    """Usuwa śmieci dodawane przez Genius (nagłówek strony, stopka 'Embed' itp.)."""
    m = HEADER_RE.search(raw_lyrics)
    text = raw_lyrics[m.start():] if m else raw_lyrics
    text = re.sub(r"\d*Embed\s*$", "", text.strip())
    text = text.replace("You might also like", "")
    return text


def parse_header(header, default_artist):
    """Z nagłówka sekcji wyciąga typ sekcji oraz listę wykonawców."""
    if ":" in header:
        section_part, artists_part = header.split(":", 1)
    else:
        section_part, artists_part = header, ""

    # "Zwrotka 2" -> "zwrotka"
    section_type = re.sub(r"[\d\s]+$", "", section_part.strip()).lower()

    artists_part = artists_part.strip()
    if artists_part:
        artists = [a.strip() for a in ARTIST_SPLIT_RE.split(artists_part) if a.strip()]
        # Usuwamy dopiski w nawiasach, np. "Mata (echo)"
        artists = [re.sub(r"\(.*?\)", "", a).strip() for a in artists]
        artists = [a for a in artists if a]
    else:
        artists = [default_artist]

    return section_type, artists


def split_into_verses(song: dict) -> list[dict]:
    """Dzieli utwór na sekcje i zwraca zwrotki z przypisanymi wykonawcami."""
    lyrics = clean_lyrics(song["lyrics"])
    default_artist = song["primary_artist"]
    headers = list(HEADER_RE.finditer(lyrics))

    verses = []
    for i, m in enumerate(headers):
        start = m.end()
        end = headers[i + 1].start() if i + 1 < len(headers) else len(lyrics)
        text = lyrics[start:end].strip()

        section_type, artists = parse_header(m.group(1), default_artist)
        if not any(section_type.startswith(keep_section) for keep_section in KEEP_SECTIONS):
            continue
        if len(text.split()) < MIN_WORDS_PER_VERSE:
            continue

        verses.append({
            "song": song["title"],
            "section": section_type,
            "text": text,
            "label": ", ".join(sorted(artists)),
        })
    return verses

# 🏷️ Budowanie zbioru danych z etykietami – agregacja i czyszczenie

Mając już pojedyncze zwrotki, przekształcamy je w końcowy zestaw danych gotowy do uczenia maszynowego.

**Krok po kroku:**

1. **Łączenie w DataFrame** – przechodzimy przez wszystkie utwory w `all_songs` i dla każdego wywołujemy `split_into_verses()`, a wyniki zbieramy w jeden `pd.DataFrame`.

2. **Deduplikacja** – `.drop_duplicates(subset=["text"])` usuwa identycznie powtarzające się zwrotki (np. ten sam refren występujący wielokrotnie w różnych częściach piosenki).

3. **Normalizacja etykiet** – `normalize_label()` aliasuje nazwy artystów: "Bedoes 2115" → "Bedoes", "Avi (POL)" → "Avi". Dodatkowo sortuje wykonawców alfabetycznie, co zapewnia spójność (np. "Mata, Quebonafide" zamiast "Quebonafide, Mata").

4. **Odrzucenie rzadkich klas** – etykiety występujące mniej niż 10 razy (`MIN_VERSES_PER_LABEL = 10`) są usuwane.

5. **Odrzucenie jednoliterowych etykiet** – pozbywamy się artefaktów, gdzie parsowanie zwróciło tylko pojedyncze znaki.

**Efekt końcowy:** czysty DataFrame z kolumnami `song`, `section`, `text`, `label`. Wypisujemy liczbę zwrotek przed i po filtrowaniu, liczbę unikalnych klas oraz rozkład etykiet – to pozwala ocenić, czy zbiór jest zbalansowany.

In [6]:
import pandas as pd

LABEL_ALIASES = {
    "Bedoes 2115": "Bedoes",
    "Avi (POL)": "Avi",
}


def normalize_label(label):
    artists = [LABEL_ALIASES.get(a.strip(), a.strip()) for a in label.split(",")]
    return ", ".join(sorted(set(artists)))


records = []
for song in all_songs:
    records.extend(split_into_verses(song))

df = pd.DataFrame(records).drop_duplicates(subset=["text"])
df["label"] = df["label"].map(normalize_label)
print(f"Liczba zwrotek przed filtrowaniem: {len(df)}")

# Odrzucamy rzadkie etykiety (np. pojedyncze gościnne wersy)
MIN_VERSES_PER_LABEL = 10
label_counts = df["label"].value_counts()
valid_labels = label_counts[label_counts >= MIN_VERSES_PER_LABEL].index
df = df[df["label"].isin(valid_labels)].reset_index(drop=True)

# Odrzucamy rekordy z jednoliterowym labelem
df = df[df["label"].str.len() > 1].reset_index(drop=True)

print(f"Liczba zwrotek po filtrowaniu: {len(df)}")
print(f"Liczba klas (etykiet): {df['label'].nunique()}\n")
print(df["label"].value_counts().to_string())

Liczba zwrotek przed filtrowaniem: 3701
Liczba zwrotek po filtrowaniu: 3090
Liczba klas (etykiet): 24

label
Sobel                  237
Mata                   208
Young Igi              199
Bedoes                 197
Zeamsone               187
White 2115             184
Malik Montana          183
Sentino                181
Kabe                   172
Quebonafide            171
Żabson                 161
Young Leosia           155
Pezet                  153
OKI                    145
Guzior                 144
Diho                   135
Avi                    109
OG Olgierd              53
Oki                     31
Louis Villain           26
Kaz Bałagane            19
Young Leosia, bambi     17
Oki, Sobel              12
Miszel                  11


# 📊 Podział na zbiór treningowy i testowy

Ostatni etap przygotowania danych – dzielimy zestaw na dwie części:

- **Zbiór treningowy (85%)** – na nim model będzie się uczyć rozpoznawać, który artysta napisał daną zwrotkę.
- **Zbiór testowy (15%)** – na nim będziemy oceniać, jak dobrze model radzi sobie z niewidzianymi wcześniej przykładami.

Używamy `train_test_split` z biblioteki `scikit-learn` z dwoma kluczowymi parametrami:

- **`stratify=df["label"]`** – zapewnia, że proporcje klas są identyczne w obu zbiorach. Jeśli np. "Mata" stanowi 20% wszystkich zwrotek, to będzie stanowił ~20% zarówno w train, jak i w test. Bez stratyfikacji mogłoby się zdarzyć, że jakaś klasa w ogóle nie trafi do zbioru testowego.
- **`random_state=42`** – ziarno losowości gwarantuje powtarzalność: za każdym uruchomieniem dostaniemy ten sam podział.

Wynik zapisujemy do plików JSON:
- `dataset/train.json` – zbiór treningowy
- `dataset/test.json` – zbiór testowy

Na koniec wyświetlamy po jednej przykładowej zwrotce z każdego zbioru, żeby zobaczyć, z jakimi danymi będzie pracował model.

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label"], # Dzielimy ze stratyfikacją - zachowujemy proporcje klas w obu zbiorach
)

train_df.to_json(os.path.join(DATASET_DIR, "train.json"), orient="records", force_ascii=False, indent=2)
test_df.to_json(os.path.join(DATASET_DIR, "test.json"), orient="records", force_ascii=False, indent=2)

print(f"Zbiór treningowy: {len(train_df)} zwrotek")
print(f"Zbiór testowy: {len(test_df)} zwrotek")

# Przykładowe zwrotki z etykietą
example = train_df.iloc[0]
print(f"\n--- Przykład ---\nUtwór: {example['song']}\nEtykieta: {example['label']}\n\n{example['text'][:500]}\n")
example = train_df.iloc[1]
print(f"\n--- Przykład ---\nUtwór: {example['song']}\nEtykieta: {example['label']}\n\n{example['text'][:500]}\n")

Zbiór treningowy: 2626 zwrotek
Zbiór testowy: 464 zwrotek

--- Przykład ---
Utwór: Bajlando
Etykieta: Miszel

Bajlando z moją bandą, robimy ten sos (cały dzień i całą noc), cały czas jest loco
Bajlando z moją bandą, robimy ten sos (cały dzień i całą noc), nie poddaj się emocjom
Bajlando z moją bandą, robimy ten sos (cały dzień i całą noc), droga będzie prostą
Bailando z moją bandą, robimy ten sos (cały dziеń i całą noc), bessa będzie hossą


--- Przykład ---
Utwór: Trzy Butelki
Etykieta: White 2115

Moja muza gra na bando
Mówią o mnie żarty, ale mnie nie śmieszą
Moja muza gra ci w duszy
Jeżeli mnie nie lubisz to się pierdol
UA, przy kompozycji braw ja umieram jako pierwszy
UA, nie chcę jej znać nienawidzę mieć depresji
UA, jestem sam no i ze mną trzy butelki
UA, nie mogę spać, powiedz gdzie moje tabletki

